# Exploratory Data Analysis — IPL Dataset
Uses Pandas and NumPy for descriptive statistics, correlation, trends, category analysis, segmentation, and pattern detection.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
PROJECT_ROOT=Path('..') if Path('../Dataset').exists() else Path('/home/user/Data-Analytics-Project')
df=pd.read_csv(PROJECT_ROOT/'Dataset/Cleaned_Data.csv',parse_dates=['match_date'])
df.shape

## Dataset overview
- Rows: **260,920**
- Matches: **1,095**
- Seasons: **17**
- Teams: **15**
- Runs: **347,756**
- Wickets: **12,950**

In [ ]:
kpis={
'Total Matches':df.match_id.nunique(),'Total Runs':df.total_runs.sum(),'Total Wickets':df.is_wicket.sum(),'Teams':pd.concat([df.batting_team,df.bowling_team]).nunique(),'Boundary %':round(df.is_boundary.mean()*100,2),'Dot Ball %':round(df.is_dot_ball.mean()*100,2),'Extras %':round(df.is_extra.mean()*100,2)}
pd.Series(kpis).to_frame('Value')

## Correlation analysis
Boundary events positively relate to total runs; dot balls are a pressure indicator; extras show controllable leakage.

In [ ]:
cols=['inning','over_number','ball','batsman_runs','extra_runs','total_runs','is_wicket','legal_ball','is_boundary','is_dot_ball','is_extra','toss_match_winner_flag']
df[cols].corr(numeric_only=True).round(3)

## Trend analysis
Season/year trends show tactical evolution and scoring inflation.

In [ ]:
season=df.groupby('match_year').agg(matches=('match_id','nunique'),runs=('total_runs','sum'),wickets=('is_wicket','sum'),balls=('legal_ball','sum'),boundaries=('is_boundary','sum'),dots=('is_dot_ball','sum')).reset_index()
season['run_rate']=season.runs/(season.balls/6)
season.tail(10)

## Category and segment analysis
Team, phase, venue, and player segments identify where performance is created or lost.

In [ ]:
team=df.groupby('batting_team').agg(runs=('total_runs','sum'),wickets_lost=('is_wicket','sum'),boundaries=('is_boundary','sum'),dots=('is_dot_ball','sum'),balls=('legal_ball','sum'),matches=('match_id','nunique')).reset_index()
team['run_rate']=team.runs/(team.balls/6); team['dot_ball_pct']=team.dots/(team.balls+1e-9)*100
team.sort_values('runs',ascending=False).head(15)

## Pattern detection
Top run scorer: **V Kohli**. Leading wicket taker: **YS Chahal**. Top winning team: **Mumbai Indians**.

In [ ]:
batters=df.groupby('batter').agg(runs=('batsman_runs','sum'),balls=('legal_ball','sum'),fours=('is_four','sum'),sixes=('is_six','sum'),matches=('match_id','nunique')).reset_index()
batters['strike_rate']=batters.runs/(batters.balls+1e-9)*100
batters.sort_values('runs',ascending=False).head(15)